# Imports

In [ ]:
import pandas as pd
import geopandas as gpd

# Load LAD data

In [ ]:
uk = gpd.read_file('uk_LADs.json')

# Load constituency data

In [ ]:
parl = gpd.read_file('uk_constituencies.json')

# Harmonise co-ordinate reference systems

In [ ]:
parl = parl.set_crs('EPSG:27700', allow_override=True).to_crs(uk.crs)

In [ ]:
parl = parl.rename(columns={'PCON24CD': 'constituency_code', 'PCON24NM': 'constituency_name'})[['constituency_code', 'constituency_name', 'geometry']]

# Calculate absolute overlap

In [ ]:
# Spatial intersection
overlap = gpd.overlay(uk, parl, how='intersection')

In [ ]:
overlap['area_overlap'] = overlap.area

# Calculate fractional overlaps

In [ ]:
constituency_areas = parl.set_index('constituency_code').geometry.area
overlap['constituency_area'] = overlap['constituency_code'].map(constituency_areas)

In [ ]:
geo_areas = uk.set_index('geo_code').geometry.area
overlap['geo_area'] = overlap['geo_code'].map(geo_areas)

In [ ]:
overlap['overlap_constituency_frac'] = overlap['area_overlap'] / overlap['constituency_area']
overlap['overlap_geo_frac'] = overlap['area_overlap'] / overlap['geo_area']

# Drop redundant columns

In [ ]:
cols = ['geo_code', 'constituency_code', 'constituency_name', 'overlap_constituency_frac', 'overlap_geo_frac']
df = overlap[cols]

# Normalise fractional overlaps

In [ ]:
df['overlap_constituency_frac_norm'] = df.groupby('constituency_code')['overlap_constituency_frac'].transform(lambda x: x/x.sum())
df['overlap_geo_frac_norm'] = df.groupby('geo_code')['overlap_geo_frac'].transform(lambda x: x/x.sum())

# Write to disk

In [ ]:
df.sort_values('constituency_code').to_csv('constituency_lad_map.csv', index=False)